In [ ]:
from fredapi import Fred
import pandas as pd
#import os
import keyring#, getpass
#import json
#import seaborn as sns
#import matplotlib.pyplot as plt

import git
from pathlib import Path

In [ ]:
#list of fred series ID's from documentation/fred_ids.txt


['DEXUSEU', 'DEXJPUS', 'DEXUSUK']

In [ ]:

def get_project_root() -> Path:
    repo = git.Repo(".", search_parent_directories=True)
    return Path(repo.working_tree_dir)

repo_root = get_project_root()

fred_ids_path = repo_root / "documentation" / "fred_ids.txt"
print(fred_ids_path)

with open(fred_ids_path, "r") as f:
    fred_series_ids = [line.strip() for line in f.readlines()]

print(fred_series_ids[:3])


/home/djmead/Documents/Projects/Financial/documentation/fred_ids.txt


['DEXUSEU', 'DEXJPUS', 'DEXUSUK']

In [3]:
todays_date = pd.Timestamp.today()

In [4]:
date_dict = {"D": pd.Timedelta(days=1),
             "W": pd.Timedelta(weeks=1),
             "M": pd.Timedelta(days=30),
             "Q": pd.Timedelta(days=90),
             "Y": pd.Timedelta(days=365),
             }

In [5]:
from db.pgsql import pgsql

In [6]:
unrate_sql_info = pgsql(table_name="UNRATE")

In [7]:
unrate_sql_info.get_sql_description()

[{'name': 'Date',
  'type': TIMESTAMP(),
  'nullable': True,
  'default': None,
  'primary_key': False},
 {'name': 'UNRATE',
  'type': DOUBLE_PRECISION(precision=53),
  'nullable': True,
  'default': None,
  'primary_key': False}]

In [8]:
unrate_sql_info.get_df_from_sql()

,Date,UNRATE
0,1990-01-01,5.4
1,1990-02-01,5.3
2,1990-03-01,5.2
3,1990-04-01,5.4
4,1990-05-01,5.4
...,...,...
435,2026-04-01,4.3
436,2026-05-01,4.3
437,2026-06-01,4.2
438,2026-07-01,4.1


In [9]:
unrate_latest_stored_date = unrate_sql_info.get_last_stored_date(date_column_name = "Date")

DatabaseError: Execution failed on sql 'SELECT MAX(Date) as last_stored_date FROM "UNRATE"': (psycopg.errors.UndefinedColumn) column "date" does not exist
LINE 1: SELECT MAX(Date) as last_stored_date FROM "UNRATE"
                   ^
HINT:  Perhaps you meant to reference the column "UNRATE.Date".
[SQL: SELECT MAX(Date) as last_stored_date FROM "UNRATE"]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [2]:
service = "fred"
api_key = keyring.get_password(service, "api_key")
fred = Fred(api_key=api_key)





In [3]:
unemployment_data = fred.get_series('UNRATE', observation_start='2026-01-01', observation_end='2026-07-31')
print("--- Unemployment Rate Series ---")
print(unemployment_data.tail(24))

--- Unemployment Rate Series ---
2026-01-01    4.3
2026-02-01    4.4
2026-03-01    4.3
2026-04-01    4.3
2026-05-01    4.3
2026-06-01    4.2
2026-07-01    4.1
dtype: float64


In [4]:
print(type(unrate_info))
print(unrate_info)

NameError: name 'unrate_info' is not defined

In [ ]:
unrate_info.to_dict()

{'id': 'UNRATE',
 'realtime_start': '2026-09-04',
 'realtime_end': '2026-09-04',
 'title': 'Unemployment Rate',
 'observation_start': '1948-01-01',
 'observation_end': '2026-08-01',
 'frequency': 'Monthly',
 'frequency_short': 'M',
 'units': 'Percent',
 'units_short': '%',
 'seasonal_adjustment': 'Seasonally Adjusted',
 'seasonal_adjustment_short': 'SA',
 'last_updated': '2026-09-04 08:27:31-05',
 'popularity': '96',
 'notes': "The unemployment rate represents the number of unemployed as a percentage of the labor force. Labor force data are restricted to people 16 years of age and older, who currently reside in 1 of the 50 states or the District of Columbia, who do not reside in institutions (e.g., penal and mental facilities, homes for the aged), and who are not on active duty in the Armed Forces.  This rate is also defined as the U-3 measure of labor underutilization.  The series comes from the 'Current Population Survey (Household Survey)'  The source code is: LNS14000000"}

In [ ]:
def get_latest(series_id):
    return fred.get_series_latest_release(series_id)

In [ ]:
class fred_getter:
    def __init__(self, api_key):
        service = "fred"
        api_key = keyring.get_password(service, "api_key")
        self.fred = Fred(api_key=api_key)
        self.series_list = []
        with open("../documentation/fred_ids.txt", "r") as file:
            self.series_list = [line.strip() for line in file.readlines() if line.strip() > 0]

    def get_series(self, series_id):
        return self.fred.get_series(series_id)

    def get_series_start_end(self, series_id, observation_start=None, observation_end=None):
        return self.fred.get_series(series_id, observation_start=observation_start, observation_end=observation_end)

    